In [17]:
# cell 1

# Add src directory to path
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

In [18]:
# cell 2

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from models import get_models
from evaluation import evaluate_model

np.random.seed(42)


In [19]:
# cell 3

# Column names for Adult dataset
columns = [
"age","workclass","fnlwgt","education","education-num",
"marital-status","occupation","relationship","race","sex",
"capital-gain","capital-loss","hours-per-week","native-country","income"
]

# Load dataset files
df_train = pd.read_csv("../data/adult/adult.data",
                       names=columns,
                       na_values=" ?",
                       skipinitialspace=True)

df_test = pd.read_csv("../data/adult/adult.test",
                      names=columns,
                      na_values=" ?",
                      skiprows=1,
                      skipinitialspace=True)

# Combine both files
df = pd.concat([df_train, df_test], ignore_index=True)

# Remove missing values
df = df.dropna()

print("Dataset shape:", df.shape)

df.head()


Dataset shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [20]:
# cell 4

# Remove trailing period in test labels
df["income"] = df["income"].str.replace(".", "", regex=False)

# Encode all categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

# Split features and target
X = df.drop("income", axis=1).values
y = df["income"].values

print("Feature matrix:", X.shape)


Feature matrix: (48842, 14)


C:\Users\Garima\AppData\Local\Temp\ipykernel_32184\3159969861.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns


In [21]:
# cell 5

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (39073, 14)
Test size: (9769, 14)


In [22]:
# cell 6

models = get_models()

results = []

for name, model in models.items():

    acc, f1 = evaluate_model(model, X_train, y_train, X_test, y_test)

    results.append([name, acc, f1])

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1"])

results_df


[LightGBM] [Info] Number of positive: 9349, number of negative: 29724
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002433 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 715
[LightGBM] [Info] Number of data points in the train set: 39073, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.239270 -> initscore=-1.156685
[LightGBM] [Info] Start training from score -1.156685


C:\Users\Garima\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,Accuracy,F1
0,Logistic Regression,0.823830,0.551940
1,Decision Tree,0.815539,0.617734
2,Random Forest,0.855973,0.672562
3,SVM,0.851162,0.640277
4,KNN,0.826594,0.615000
5,LightGBM,0.878800,0.724395
